In [288]:
import pandas
import pathlib

In [289]:
files = list(pathlib.Path('.').glob('data/*.csv'))

In [290]:
base_columns = set(
    [
        'title',
        'name',
        'age',
        'height',
        'elected_in',
        'place_of_residence',
        'result',
        'ethnicity',
        'hair_color',
        'country',
        'competition'
    ]
)

## Columns

Each file come with different columns. We need to ensure that the final outputs contains the same columns

In [291]:
dfs = [pandas.read_csv(f) for f in files]

In [292]:
for item in dfs:
    df_columns = set(item.columns)
    missing_columns = base_columns - df_columns
    for col in missing_columns:
        if col == 'ethnicity':
            item[col] = 'W'
            continue

        if col == 'hair_color':
            item[col] = 'D'
            continue

        if col == 'height':
            item[col] = 0
            continue

        item[col] = pandas.NA

In [293]:
df = pandas.concat(dfs, ignore_index=True)

In [294]:
df.head()

,country,name,age,elected_in,year,competition,result,height,place_of_residence,ethnicity,title,hair_color
0,Angola,Adalgisa Gonçalves,21,Luanda,2001,Miss World,NaN,0,NaN,W,NaN,D
1,Antigua and Barbuda,Janelle Williams,23,Saint John,2001,Miss World,NaN,0,NaN,W,NaN,D
2,Argentina,Virginia di Salvo,22,Rosario,2001,Miss World,NaN,0,NaN,W,NaN,D
3,Aruba,Zizi Lee,19,Oranjestad,2001,Miss World,1RU,0,NaN,W,NaN,D
4,Australia,Eva Milic,23,Gold Coast,2001,Miss World,NaN,0,NaN,W,NaN,D


## Geocode the cities

In [295]:
fr_post_codes_path = pathlib.Path('..').joinpath('geo/fr_code_postal_v2.csv').absolute()

In [296]:
fr_post_codes = pandas.read_csv(fr_post_codes_path)

In [297]:
fr_post_codes = fr_post_codes[['commune', 'gps']]

In [298]:
fr_post_codes.commune = fr_post_codes.commune.apply(lambda x: x.lower())

In [299]:
french_pageants = df[df.country.str.contains('rance', na=False)]
french_pageants.country.count()

np.int64(45)

In [300]:
for pageant in french_pageants.itertuples(name='Pageant'):
    municipality = fr_post_codes[fr_post_codes.commune == pageant.elected_in.lower()]
    df.loc[pageant.Index, 'municipality'] = municipality.commune.values[-1] if not municipality.empty else None
    df.loc[pageant.Index, 'municipality_gps'] = municipality.gps.values[-1] if not municipality.empty else None

## Geocode countries

- ICU
- Languages

## Additional infos

In [301]:
df['birth_year'] = df['year'] - df['age']

In [302]:
df['elected_in_wikidata'] = pandas.NA

In [303]:
df.to_csv('pageants.csv', index=False)

In [304]:
elected_in_cities = df[~df.elected_in.duplicated() & df.elected_in_wikidata.isna()][['elected_in', 'elected_in_wikidata']]

In [305]:
elected_in_cities.sort_values('elected_in', inplace=True)

In [306]:
elected_in_cities.to_csv('elected_in_cities_to_geocode.csv', index=False)